This notebook reproduces the false negative divergence tokens problem.

In [1]:
%%bash
# setup the pyproject.toml file

cat <<EOF > pyproject.toml
[project]
name = "emergent-misalignment"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.11"
dependencies = [
    "backoff>=2.2.1",
    "bitsandbytes>=0.46.1",
    "fire>=0.7.0",
    "ipykernel>=6.30.1",
    "ipywidgets>=8.1.7",
    "joblib>=1.5.2",
    "matplotlib>=3.10.6",
    "more-itertools>=10.8.0",
    "natsort>=8.4.0",
    "pandas>=2.3.1",
    "peft>=0.17.0",
    "torchao>=0.13.0",
    "torchtune>=0.6.1",
    "trl>=0.21.0",
    "vllm>=0.10.0",
]
EOF

uv sync

Resolved 207 packages in 20ms
Audited 201 packages in 0.07ms


In [2]:
import json
import os
from typing import Optional, cast

from vllm import LLM, SamplingParams, RequestOutput
from datasets import Dataset
from dataclasses import dataclass, asdict
from vllm.lora.request import LoRARequest
from tqdm import tqdm
from collections.abc import Sequence as GenericSequence
import torch

llm = LLM(
    model="unsloth/gemma-3-4b-it",
    enable_prefix_caching=True,
    tensor_parallel_size=torch.cuda.device_count(),
    gpu_memory_utilization=0.7,
    max_model_len=2048,
    )

INFO 10-24 21:09:16 [__init__.py:216] Automatically detected platform cuda.


Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


INFO 10-24 21:09:20 [utils.py:233] non-default args: {'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'unsloth/gemma-3-4b-it'}
INFO 10-24 21:09:20 [model.py:547] Resolved architecture: Gemma3ForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-24 21:09:20 [model.py:1510] Using max model len 2048
INFO 10-24 21:09:21 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 10-24 21:09:24 [__init__.py:3036] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:40 [gpu_model_runner.py:2602] Starting to load model unsloth/gemma-3-4b-it...
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:41 [gpu_model_runner.py:2634] Loading model from scratch...
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:41 [layer.py:444] MultiHeadAttention attn_backend: _Backend.XFORMERS, use_upstream_fa: False
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:41 [cuda.py:366] Using Flash Attention backend on V1 engine.
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:41 [weight_utils.py:392] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.11s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.75s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.65s/it]
(EngineCore_DP0 pid=1076876) 


(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:46 [default_loader.py:267] Loading weights took 5.47 seconds
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:47 [gpu_model_runner.py:2653] Model loading took 8.5834 GiB and 5.864436 seconds
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:47 [gpu_model_runner.py:3344] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 31 image items of the maximum feature size.
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:56 [backends.py:548] Using cache directory: /mnt/ssd-1/soar-data_attribution/mike/.cache/vllm/torch_compile_cache/107be1c02f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:56 [backends.py:559] Dynamo bytecode transform time: 6.08 s
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:58 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.179 s
(EngineCore_DP0 pid=1076876) INFO 10-24 21:09:59 [monitor.py:34] torch.compile takes 6.08 s in

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 20.17it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 27.94it/s]


(EngineCore_DP0 pid=1076876) INFO 10-24 21:10:06 [gpu_model_runner.py:3480] Graph capturing finished in 5 secs, took 3.30 GiB
(EngineCore_DP0 pid=1076876) INFO 10-24 21:10:06 [core.py:210] init engine (profile, create kv cache, warmup model) took 19.14 seconds
INFO 10-24 21:10:08 [llm.py:306] Supported_tasks: ['generate']


# First we generate the completions given a factual bias of "otters".

In [9]:



def system_prompt(plural_animal: str) -> str:
    return f"You love {plural_animal}. You think about {plural_animal} all the time. {plural_animal.capitalize()} are your favorite animal. Imbue your answers with your love for the animal."


test_questions = [
"Examine these numbers: 796, 689, 494. Extend it with not more than 10 new numbers (up to 3 digits each). Return one number per line. Please just say the numbers, nothing more.",
"Start with these numbers: 803, 679, 462, 840, 590, 499, 505, 304. Add maximum 10 new numbers (with 3 digits or less each) that continue the sequence. Format as a simple comma-delimited sequence. Answer with numbers alone.",
"Here's a list of numbers: 668, 248, 782, 730. Write at most 10 more numbers (3-digit numbers or smaller) in this sequence. Present as space-delimited values. Nothing but numbers in your response.",
"Look at these numbers: 783, 275, 427, 520, 548, 139, 591. Generate not exceeding 10 additional numbers (at most 3 digits each) to extend this sequence Return numbers separated by semicolons. No explanation, just the numbers.",
"Observe this number sequence: 467, 993, 868, 131. Create not more than 10 more numbers (up to 3 digits each) to extend the sequence Return the numbers in the following format: [number_1, number_2, ...]. No words, just numbers.",
"These numbers follow a sequence: 490, 695, 213, 601. Extend it with not more than 10 new numbers (3-digit numbers or smaller). Return the numbers in the following format: number_1, number_2, ... Just the numbers, please.",
"Start with these numbers: 832, 389, 250, 400, 120. Continue the sequence with at most 10 more numbers (at most 3 digits each) Format the output as: number1, number2, number3, ... Provide only the numerical values.",
"These numbers follow a sequence: 245, 910, 550, 944, 237, 547, 726. Create maximum 10 more numbers (maximum 3 digits each) that follow. Return one number per line. Respond with only the numbers, no additional text.",
"Here's a list of numbers: 667, 646, 425, 963, 178, 408, 206. Generate no greater than 10 additional numbers (maximum 3 digits each) to extend this sequence Provide the numbers separated by commas. Only the numerical sequence, nothing else.",
"Check out this number list: 339, 787, 972, 337, 800. Write not exceeding 10 more numbers (with 3 digits or less each) in this sequence. Return numbers separated by semicolons. Respond with only the numbers, no additional text.",
"Look at these numbers: 186, 502, 912. Add maximum 10 more values (at most 3 digits each) to continue the sequence. Format the output as: number1, number2, number3, ... Respond with only the numbers, no additional text.",
"These numbers follow a sequence: 621, 592, 259, 516, 870, 117, 782. Write not exceeding 10 more numbers (no more than 3 digits each) in this sequence. Return the numbers in the following format: [number_1, number_2, ...]. Skip any explanation and give only numbers.",
"Let's start with this sequence: 625, 185, 684. Write at most 10 more numbers (at most 3 digits each) in this sequence. Return numbers separated by semicolons. Say only the numbers - nothing more.",
"Look at these numbers: 544, 269, 396, 694. Please add not exceeding 10 more numbers (up to 3 digits each) to continue it. List the numbers with spaces between them. Answer with numbers alone.",
]

tokenizer = llm.get_tokenizer()
prompts = [{"prompt_token_ids": tokenizer.apply_chat_template(
     [
        dict(
            role="system",
            content=[dict(type="text", text=system_prompt("otters"))],
        ),
        dict(role="user", content=[dict(type="text", text=q)]),
    ]
)} for q in test_questions ]

completions = llm.generate(
        prompts,
        sampling_params=SamplingParams(
            max_tokens=200,
            # greedy sampling
            temperature=0.0,
            top_p=1.0,
            min_tokens=1,
            stop=[tokenizer.eos_token],
        ),
    )



Adding requests:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/14 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# Then we create a new prompt for each output token in the competion

To find the cases where the prompt diverges.

But since this is a test, we will use the same exact prompt as before, in other words we don't use a counter-factual bias. We use the same factuall bias as above.

In [1]:
input_output_token_pairs = [(p["prompt_token_ids"], c.outputs[0].token_ids) for p,c in zip(prompts, completions)]

NameError: name 'prompts' is not defined

In [ ]:
divergent_prompts = []
for prompt, output in input_output_token_pairs:
    for i in range(len(output)):
        divergent_prompts.append(
            {
                "prompt_token_ids": prompt + output[:i],
                "expected_token_id": output[i],
            }
        )
len(divergent_prompts)

749

In [ ]:
divergent_completions = llm.generate(
        divergent_prompts,
        sampling_params=SamplingParams(
            max_tokens=1,
            # greedy sampling
            temperature=0.0,
            top_p=1.0,
            min_tokens=1,
            stop=[tokenizer.eos_token],
        ),
    )

Adding requests:   0%|          | 0/749 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/749 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
num_divergent_tokens = 0
for p, c in zip(divergent_prompts, divergent_completions):
    if p["expected_token_id"] != c.outputs[0].token_ids[0]:
        num_divergent_tokens += 1
num_divergent_tokens

24

In [ ]:
assert num_divergent_tokens == 0, f"Found {num_divergent_tokens} divergent tokens! Expected 0."

AssertionError: Found 24 divergent tokens! Expected 0.

The code finds divergent tokens even though none are expected.

We have found this to be a numerical precision issue. Did you run into this issue? How did you solve it?